# Iceberg Lakehouse + Elasticsearch Search

**Routing pattern:**
- WRITE: `[iceberg, elasticsearch]` — fan-out; Iceberg is primary (ACID snapshots), ES synced async
- READ: `[iceberg]` — browse/paginate via PyIceberg scan
- SEARCH: `[elasticsearch]` — bbox, attribute, fulltext queries
- METADATA: `[postgresql]` — collection metadata in PG

Best for versioned analytical datasets where time-travel and lakehouse integration matter.

In [ ]:
import httpx

BASE = "http://localhost:8000"
CATALOG_ID = "demo-lakehouse"
COLLECTION_ID = "flood-observations"

client = httpx.AsyncClient(base_url=BASE, timeout=30)
print("Client ready")

In [ ]:
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Lakehouse Demo"})
print(r.status_code)

In [ ]:
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Flood Observations",
    "description": "Near-real-time flood extent polygons — Iceberg primary with ES for fast bbox search",
    "license": "CC0-1.0",
})
print(r.status_code)

In [ ]:
routing = {
    "plugin_id": "storage:collections",
    "operations": {
        "WRITE": [
            {"driver_id": "iceberg", "on_failure": "fatal", "write_mode": "sync"},
            {"driver_id": "elasticsearch", "on_failure": "warn", "write_mode": "async"}
        ],
        "READ": [{"driver_id": "iceberg"}],
        "SEARCH": [{"driver_id": "elasticsearch"}],
        "METADATA": [{"driver_id": "postgresql"}]
    }
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/storage:collections",
    json=routing,
)
print(r.status_code)

In [ ]:
# NEW_VERSION policy: every re-ingestion appends a new Iceberg snapshot
policy = {
    "plugin_id": "write_policy",
    "on_conflict": "new_version",
    "track_asset_id": True,
    "enable_validity": True,
    "external_id_field": "id"
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/write_policy",
    json=policy,
)
print(r.status_code)

In [ ]:
import datetime

# Version 1 of flood observations (initial extent)
features_v1 = [
    {
        "type": "Feature",
        "id": f"flood-{i:03d}",
        "geometry": {"type": "Point", "coordinates": [14.0 + i * 0.1, 40.5 + i * 0.1]},
        "properties": {
            "severity": ["low", "medium", "high"][i % 3],
            "water_depth_m": round(0.5 + i * 0.3, 2),
            "observed_at": "2024-01-15T06:00:00Z",
        }
    }
    for i in range(5)
]
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features_v1},
)
print(r.status_code, "Version 1 written to Iceberg (snapshot 1)")

In [ ]:
# Version 2 — same IDs, updated severity (NEW_VERSION creates a 2nd Iceberg snapshot)
features_v2 = [
    {
        "type": "Feature",
        "id": f"flood-{i:03d}",
        "geometry": {"type": "Point", "coordinates": [14.0 + i * 0.1, 40.5 + i * 0.1]},
        "properties": {
            "severity": "high",  # escalated
            "water_depth_m": round(1.5 + i * 0.3, 2),  # deeper
            "observed_at": "2024-01-15T12:00:00Z",
        }
    }
    for i in range(5)
]
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features_v2},
)
print(r.status_code, "Version 2 written to Iceberg (snapshot 2 — time-travel available)")

In [ ]:
# READ from Iceberg — returns latest snapshot (both v1 and v2 rows due to NEW_VERSION)
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=20")
data = r.json()
print(f"Iceberg READ: {len(data.get('features', []))} items (v1 + v2)")

In [ ]:
# SEARCH via ES — bbox filter (only latest indexed versions)
r = await client.post(
    f"/search/catalogs/{CATALOG_ID}/items-search",
    json={
        "collections": [COLLECTION_ID],
        "bbox": [13.9, 40.4, 14.6, 41.0],
        "filter": {"op": "=", "args": [{"property": "severity"}, "high"]},
        "limit": 10,
    },
)
data = r.json()
print(f"ES SEARCH (high severity in bbox): {len(data.get('features', []))} results")